# 02 – Data Cleaning and Quality Control

## 1. Objective

This notebook performs the data cleaning and quality control stage of the project pipeline. It builds on the raw GDELT event table created in the ingestion notebook and prepares a cleaner event-level dataset for the later Narrative Volatility Index (NVI) construction.

The cleaning process focuses on:

- inspecting the raw imported event table,
- identifying missing values across variables,
- removing columns with extremely high missingness,
- filtering records to the intended study period,
- handling missing country identifiers,
- checking duplicate event identifiers,
- validating numerical variables used in the NVI analysis,
- reviewing event class and event type distributions,
- creating a cleaned table for country-month aggregation.

The output of this notebook is a cleaned DuckDB table named `exports_main_clean`, which serves as the main input for the NVI construction notebook.

## 2. Database Connection

The cleaned dataset is created from the `exports_main` table stored in the local DuckDB database. The connection is established first, and the available tables are inspected to confirm that the raw event table is accessible.

In [2]:
import duckdb

con = duckdb.connect("../data/intermediate/gdelt_main.db")

The available tables in the database are listed below.

In [3]:
con.execute("SHOW TABLES").fetchdf()

,name
0,country_filter
1,exports_main
2,exports_main_clean
3,narrative_country_12month
4,narrative_country_clean
5,narrative_country_full
6,narrative_country_month
7,narrative_nvi
8,narrative_volatility
9,narrative_volatility_2025


## 3. Raw Table Overview

Before cleaning, the size of the raw imported event table is checked. This provides a baseline for comparing the number of records before and after filtering.

In [4]:
con.execute("""
SELECT COUNT(*) AS raw_total_rows
FROM exports_main
""").fetchdf()

,raw_total_rows
0,50325112


## 4. Missing Value Analysis

Missing values are examined across all columns in the raw event table. This step helps identify variables that are complete, partially missing, or largely unavailable.

The missing value summary reports both the number and percentage of missing records for each column.

In [5]:
import pandas as pd

cols = con.execute("DESCRIBE exports_main").fetchdf()["column_name"].tolist()

results = []

for col in cols:
    query = f"""
    SELECT 
        '{col}' AS column_name,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(
            100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS null_percentage
    FROM exports_main
    """
    results.append(con.execute(query).fetchdf())

null_df = pd.concat(results, ignore_index=True)
null_df.sort_values("null_percentage", ascending=False)

,column_name,total_rows,null_count,null_percentage
24,Actor2Type3Code,50325112,50306746.0,99.96
14,Actor1Type3Code,50325112,50297179.0,99.94
21,Actor2Religion2Code,50325112,50194822.0,99.74
11,Actor1Religion2Code,50325112,50161218.0,99.67
19,Actor2EthnicCode,50325112,50087561.0,99.53
...,...,...,...,...
30,GoldsteinScale,50325112,261.0,0.00
32,NumSources,50325112,0.0,0.00
51,ActionGeo_Type,50325112,0.0,0.00
59,DATEADDED,50325112,0.0,0.00


The following table displays only columns that contain at least one missing value.

In [6]:
null_cols = null_df[null_df["null_count"] > 0]
null_cols.sort_values("null_percentage", ascending=False)

,column_name,total_rows,null_count,null_percentage
24,Actor2Type3Code,50325112,50306746.0,99.96
14,Actor1Type3Code,50325112,50297179.0,99.94
21,Actor2Religion2Code,50325112,50194822.0,99.74
11,Actor1Religion2Code,50325112,50161218.0,99.67
19,Actor2EthnicCode,50325112,50087561.0,99.53
9,Actor1EthnicCode,50325112,50021214.0,99.40
18,Actor2KnownGroupCode,50325112,49835620.0,99.03
20,Actor2Religion1Code,50325112,49791796.0,98.94
8,Actor1KnownGroupCode,50325112,49736163.0,98.83
10,Actor1Religion1Code,50325112,49701149.0,98.76


## 5. High-Null Column Filtering

Columns with extremely high missingness provide limited analytical value and may introduce unnecessary noise. Therefore, columns with more than 90% missing values are identified and removed from the cleaned event table.

The 90% threshold is used as a conservative rule to remove variables that are almost entirely unavailable while preserving variables that may still be useful for analysis.

In [7]:
high_null_cols = null_df[null_df["null_percentage"] > 90]["column_name"].tolist()

high_null_cols

['Actor1KnownGroupCode',
 'Actor1EthnicCode',
 'Actor1Religion1Code',
 'Actor1Religion2Code',
 'Actor1Type2Code',
 'Actor1Type3Code',
 'Actor2KnownGroupCode',
 'Actor2EthnicCode',
 'Actor2Religion1Code',
 'Actor2Religion2Code',
 'Actor2Type2Code',
 'Actor2Type3Code']

## 6. Creating the Cleaned Event Table

A cleaned event table is created by excluding columns with more than 90% missing values and filtering records to the intended study period.

The intended study period covers events from `2025-01-01` to `2026-03-26`. Records outside this range were identified during the ingestion stage and are excluded here to ensure that the final analysis is based only on the intended temporal scope.

In [8]:
expected_start = 20250101
expected_end = 20260326

columns_to_keep = [col for col in cols if col not in high_null_cols]
columns_to_keep_sql = ", ".join(columns_to_keep)

con.execute(f"""
CREATE OR REPLACE TABLE exports_main_clean AS
SELECT {columns_to_keep_sql}
FROM exports_main
WHERE Day BETWEEN {expected_start} AND {expected_end}
""")

## 7. Validation of the Cleaned Table

After creating the cleaned table, its structure and missing values are checked again. This validates that the cleaning operations were applied correctly and that the resulting table is ready for later analysis.

In [12]:
cols_info = con.execute("DESCRIBE exports_main_clean").fetchdf()
cols_info

,column_name,column_type,null,key,default,extra
0,GlobalEventID,BIGINT,YES,None,None,None
1,Day,BIGINT,YES,None,None,None
2,MonthYear,BIGINT,YES,None,None,None
3,Year,BIGINT,YES,None,None,None
4,FractionDate,DOUBLE,YES,None,None,None
5,Actor1Code,VARCHAR,YES,None,None,None
6,Actor1Name,VARCHAR,YES,None,None,None
7,Actor1CountryCode,VARCHAR,YES,None,None,None
8,Actor1Type1Code,VARCHAR,YES,None,None,None
9,Actor2Code,VARCHAR,YES,None,None,None


In [13]:
cols_clean = con.execute("DESCRIBE exports_main_clean").fetchdf()["column_name"].tolist()

results_clean = []

for col in cols_clean:
    query = f"""
    SELECT 
        '{col}' AS column_name,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(
            100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS null_percentage
    FROM exports_main_clean
    """
    results_clean.append(con.execute(query).fetchdf())

null_df_clean = pd.concat(results_clean, ignore_index=True)
null_df_clean.sort_values("null_percentage", ascending=False)

,column_name,total_rows,null_count,null_percentage
12,Actor2Type1Code,49989079,33252907.0,66.52
35,Actor2Geo_ADM2Code,49989079,31391128.0,62.80
8,Actor1Type1Code,49989079,28396622.0,56.81
11,Actor2CountryCode,49989079,27197537.0,54.41
43,ActionGeo_ADM2Code,49989079,24081165.0,48.17
27,Actor1Geo_ADM2Code,49989079,23183989.0,46.38
7,Actor1CountryCode,49989079,21438339.0,42.89
32,Actor2Geo_FullName,49989079,15043685.0,30.09
36,Actor2Geo_Lat,49989079,15043703.0,30.09
37,Actor2Geo_Long,49989079,15038207.0,30.08


## 8. Country Code Handling

Country identifiers are important for later country-level aggregation. In GDELT event data, actor country fields may contain missing values because not all actors are associated with explicit country codes.

For this project, `ActionGeo_CountryCode` is the most important country identifier because it represents the geographic location where the event action occurred. Actor country variables are inspected for completeness, but the later country-month aggregation primarily relies on action geography.

In [14]:
con.execute("""
SELECT Actor1CountryCode, COUNT(*) AS count
FROM exports_main_clean
GROUP BY Actor1CountryCode
ORDER BY count DESC
LIMIT 20;
            """).fetchdf()

,Actor1CountryCode,count
0,NaN,21438339
1,USA,8788871
2,GBR,1475386
3,ISR,1138949
4,RUS,895830
5,CHN,875468
6,CAN,780054
7,UKR,686860
8,AUS,651712
9,NGA,637924


In [15]:
con.execute("""
SELECT Actor2CountryCode, COUNT(*) AS count
FROM exports_main_clean
GROUP BY Actor2CountryCode
ORDER BY count DESC
LIMIT 20
""").fetchdf()

,Actor2CountryCode,count
0,NaN,27197537
1,USA,6302658
2,GBR,1082239
3,ISR,1062501
4,RUS,855527
5,CHN,748511
6,UKR,705905
7,PSE,601456
8,CAN,561000
9,IRN,505332


In [16]:
con.execute("""
SELECT ActionGeo_CountryCode, COUNT(*) AS count
FROM exports_main_clean
GROUP BY ActionGeo_CountryCode
ORDER BY count DESC
LIMIT 20
""").fetchdf()

,ActionGeo_CountryCode,count
0,US,16020857
1,IN,2597999
2,UK,2594500
3,IS,2229778
4,NI,1467287
5,CH,1351203
6,NaN,1342724
7,CA,1341697
8,RS,1267938
9,AS,1180484


Missing country codes are not deleted at this stage. Instead, they are inspected and may be labeled as `Unknown` when needed for descriptive summaries. For the final NVI construction, records without a valid `ActionGeo_CountryCode` should be excluded from country-level aggregation.

In [17]:
cols = [
"Actor1CountryCode",
"Actor2CountryCode",
"Actor1Geo_CountryCode",
"Actor2Geo_CountryCode",
"ActionGeo_CountryCode"
]

for c in cols:
    con.execute(f"""
        UPDATE exports_main_clean
        SET {c} = 'Unknown'
        WHERE {c} IS NULL
    """)

In [19]:
con.execute("""
SELECT ActionGeo_CountryCode, COUNT(*) AS count
FROM exports_main_clean
GROUP BY ActionGeo_CountryCode
ORDER BY count DESC
LIMIT 20
""").fetchdf()

,ActionGeo_CountryCode,count
0,US,16020857
1,IN,2597999
2,UK,2594500
3,IS,2229778
4,NI,1467287
5,CH,1351203
6,Unknown,1342724
7,CA,1341697
8,RS,1267938
9,AS,1180484


## 9. Duplicate Event Check

The `GlobalEventID` variable is expected to identify unique GDELT events. Duplicate checks are performed to verify whether the cleaned table contains repeated event identifiers.

In [20]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT GlobalEventID) AS unique_events
FROM exports_main_clean
""").fetchdf()

,total_rows,unique_events
0,49989079,49989079


## 10. Numerical Variable Range Checks

The numerical variables used later in the NVI analysis are checked for plausible ranges. This includes the Goldstein score, average tone, and media coverage variables such as mentions, sources, and articles.

In [21]:
con.execute("""
SELECT
    MIN(GoldsteinScale) AS min_goldstein,
    MAX(GoldsteinScale) AS max_goldstein,
    AVG(GoldsteinScale) AS avg_goldstein,
    MIN(AvgTone) AS min_tone,
    MAX(AvgTone) AS max_tone,
    AVG(AvgTone) AS avg_tone,
    MIN(NumMentions) AS min_mentions,
    MAX(NumMentions) AS max_mentions,
    MIN(NumSources) AS min_sources,
    MAX(NumSources) AS max_sources,
    MIN(NumArticles) AS min_articles,
    MAX(NumArticles) AS max_articles
FROM exports_main_clean
""").fetchdf()

,min_goldstein,max_goldstein,avg_goldstein,min_tone,max_tone,avg_tone,min_mentions,max_mentions,min_sources,max_sources,min_articles,max_articles
0,-10.0,10.0,0.575825,-52.941176,37.5,-1.950872,1,2800,1,48,1,2800


## 11. Event Class Distribution

The distribution of `QuadClass` is inspected because this variable is later used to compute structural entropy. `QuadClass` groups events into broad categories and helps capture variation in the type of events observed over time.

In [24]:
con.execute("""
SELECT 
    QuadClass, 
    COUNT(*) AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM exports_main_clean
GROUP BY QuadClass
ORDER BY QuadClass
""").fetchdf()

,QuadClass,count,percentage
0,1,30785713,61.58
1,2,5734441,11.47
2,3,6537776,13.08
3,4,6931149,13.87


## 12. Event Root Code Distribution

The `EventRootCode` distribution is inspected to understand the most common broad event types in the cleaned dataset. This provides additional context about the structure of the event data before aggregation.

In [25]:
con.execute("""
SELECT 
    EventRootCode, 
    COUNT(*) AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM exports_main_clean
GROUP BY EventRootCode
ORDER BY count DESC
LIMIT 20
""").fetchdf()

,EventRootCode,count,percentage
0,04,12416649,24.84
1,01,7107204,14.22
2,05,4461241,8.92
3,02,3546735,7.10
4,03,3253877,6.51
5,11,3238891,6.48
6,19,3000503,6.00
7,17,2461202,4.92
8,08,1785475,3.57
9,07,1765618,3.53


## 13. Geographic Coordinate Check

Geographic coordinates are checked to identify whether action-level latitude and longitude values are available. These variables are not the main focus of the NVI construction, but they provide additional geographic information that may be useful for later mapping or spatial analysis.

In [28]:
con.execute("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ActionGeo_Lat IS NULL OR ActionGeo_Long IS NULL THEN 1 ELSE 0 END) AS missing_action_coordinates,
    ROUND(
        100.0 * SUM(CASE WHEN ActionGeo_Lat IS NULL OR ActionGeo_Long IS NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS missing_action_coordinates_pct
FROM exports_main_clean
""").fetchdf()

,total_rows,missing_action_coordinates,missing_action_coordinates_pct
0,49989079,1360137.0,2.72


## 14. Summary

This notebook cleaned and validated the raw GDELT event table created during the ingestion stage. Columns with more than 90% missing values were identified and excluded from the cleaned dataset. Records outside the intended study period from 2025-01-01 to 2026-03-26 were also filtered out.

The resulting table, `exports_main_clean`, provides a cleaner event-level dataset for later analysis. Key event identifiers, time variables, action geography, event class information, Goldstein scores, media tone, and media coverage variables were retained because they are required for country-month aggregation and NVI construction.

The next notebook uses this cleaned event table to construct country-month narrative indicators, compute rolling volatility measures, and build the final Narrative Volatility Index.